<div align="right"><i>Matías Torres Esteban<br>Enero, 2026</i></div>

# Creación y Visualización de Mapas Conceptuales

En esta notebook veremos las utilidades y herramientas que provee nuestra librería para manipular mapas conceptuales. Podemos crear un mapa conceptual de forma manual o utilizando herramientas de extracción de información provistas en el submódulo `cmapper`. Ambos métodos de creación pueden ser combinados para obtener una base de conocimiento idiosincrática y que contiene tanto conocimiento personal como el que se encuentra en vastos cuerpos de documentos científicos. Algunas aplicaciones de estas herramientas son:

1. Sistemas de organización de conocimiento personal y similares a [Obsidian](https://obsidian.md/).
2. Sistemas de preguntas y respuestas basadas en grafos, capaces de resolver desafíos tales como [SQuaAD](https://rajpurkar.github.io/SQuAD-explorer/) [1] y [PubMedQA](https://pubmedqa.github.io/) [2].
3. Sistemas asistentes al investigador tal como el propuesto por [Markus Buhler](https://arxiv.org/abs/2403.11996) [3].

Los mapas conceptuales son una herramienta muy poderosa para representar conocimiento porque han sido validados en el ámbito de la educación y la psicología [4]. Debido a su carácter general y por ser un formalismo muy simple, es posible expresar conocimientos de diferentes dominios en el lenguaje de los mapas conceptuales. Estos tienen muchas semejanzas con los grafos de conocimiento y pueden considerarse lenguajes equivalentes. Mi premisa es que los mapas conceptuales son un primer paso en el proceso de estructuración de conocimiento, siendo posible transformarlos a lenguajes de mayor rigor en pasos subsiguientes.  

## Mapas Personales

En esta sección aprenderemos a almacenar, visualizar y eliminar un mapa conceptual creado manualmente con las herramientas que provee la librería. Nuestro primer paso será configurar los parámetros de conexión a la base de conocimiento: 

In [29]:
db_config = {'dbname': 'my_mind', 'port': 5432, 'host': 'localhost', 'password': 'postgres', 'user': 'postgres'}

El próximo paso será importar los módulos necesarios y crear los objetos orquestadores que se encargan de implementar la mayor parte de la lógica de altas y bajas. El código de creación puede parecer tedioso pero hace explícitas las dependencias de cada clase. Además, este diseño modular permite intermezclar componentes de diversos modos. 

In [30]:
from llms_kgs.persistence import (ConnectionProvider, 
    CMapInserter, CMapRetriever, CMapDeleter)

from llms_kgs.logic import CMapAdder, CMapRemover, CMapRecoverer
from llms_kgs.logic.cmap_encoder import CMapQuestionEncoder
from llms_kgs.llms import M3Encoder

# Create connection to knowledge base: 
connection_provider = ConnectionProvider(**db_config)

# Create concept map inserter: 
cmap_inserter = CMapInserter(connection_provider)

# Create concept map retriever:
cmap_retriever = CMapRetriever(connection_provider)

# Create concept map deleter:
cmap_deleter = CMapDeleter(connection_provider)

# Create sentence transformer:
m3 = M3Encoder()

# Create concept map encoder, which computes 
# meaningful embeddings for concept maps: 
cmap_encoder = CMapQuestionEncoder(m3)

# Create concept map adder, which orchestrates
# the logic for adding a concept map to the 
# knowledge base:
cmap_adder = CMapAdder(
    cmap_encoder = cmap_encoder,
    retriever = cmap_retriever,
    inserter=cmap_inserter,
    ef=m3)

# Create concept map remover, which orchestrates
# the logic for removing a concept map from the
# knowledge base:
cmap_remover = CMapRemover(
    retriever = cmap_retriever,
    deleter = cmap_deleter)

# Create concept map retriever: 
cmap_recoverer = CMapRecoverer(cmap_retriever)

A continuación crearemos un mapa conceptual propio utilizando los objetos de dominio del módulo `domain`. 

In [31]:
from llms_kgs.domain import CMap, Triple

triples = [
    Triple('Metamathematics', 'is the study of', 'Mathematics'),
    Triple('Metamathematics', 'uses', 'Mathematical Methods'),
    Triple('Metamathematics', 'produces', 'Metatheories'), 
    Triple('Metamathematics', 'has as milestone', 'Discovery of Hyberbolic Geometry'), 
    Triple('Discovery of Hyberbolic Geometry', 'occurred in', '19th Century'),
    Triple('Metatheories', 'are theories of', 'Methematical Theories'), 
]

cmap = CMap(title = 'Metamathematics', focus_question = 'What is Metamathematics?', triples = triples)

Daremos de alta nuestro mapa conceptual en la base de conocimiento con el objeto `cmap_adder`. Este realiza los siguientes pasos:

1. Valida que el mapa conceptual sea correcto (Por ejemplo, que las etiquetas de los conceptos no sean cadenas vacías).
2. Calcula el embedding del mapa conceptual con el objeto `cmap_encoder`. Aunque no lo realizaremos en esta notebook, este embedding puede utilizarse para  búsquedas semánticas sobre el espacio de mapas conceptuales. 
3. Calcula los embeddings individuales de cada terna de conocimiento.
4. Invoca al objeto `cmap_inserter` para insertar al mapa conceptual en la base de conocimiento.

El proceso devuelve un objeto de notificación el cual informa si ocurrieron errores durante el proceso. 

In [32]:
from llms_kgs.core_utils import Notification

def print_notification_errors(notification: Notification):
    """ Helper function to print error messages from a notification object """
    for error in notification.get_errors():
        print(f'- {error}')

notification = cmap_adder.add(cmap)

if(notification.has_errors()):
    print_notification_errors(notification)
    
else:
    print('Success!')

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Success!


Ahora recuperaremos el mapa conceptual insertado para verificar que la operación fue exitosa.

In [33]:
result = cmap_recoverer.recover_by_title('Metamathematics')

notification = result['notification']
cmap = result['cmap']

if(notification.has_errors()):
    print_notification_errors(notification)
    
else:
    print('Success!')

Success!


Utilizaremos la herramienta `cmap_drawer` para visualizar nuestro mapa conceptual como un grafo dirigido.

In [43]:
from IPython.display import display, HTML
from llms_kgs.logic import CMapDrawer
from pyvis.network import Network
import networkx as nx
import tempfile
import os

# Draw the concept map as a Pyvis Network:
network = CMapDrawer.draw([cmap])

# Set the height and width of the network:
network.height = "500px"
network.width = "100%"

# Prepare the notebook resources to initialize the template engine
network.prep_notebook() 

# Show the network:
network.show('network.html', notebook=True)

network.html


Bellísimo! Por último eliminaremos el mapa conceptual para revertir la base de conocimiento a su estado original.

In [12]:
notification = cmap_remover.remove_by_title('Metamathematics')

if(notification.has_errors()):
    print_notification_errors(notification)
    
else:
    print('Success!')

Success!


## Extracción Automática de Información

Podemos crear automáticamente mapas conceptuales a partir de texto usando el modulo `cmapper`. Esta manipula LLMs para interpretar un texto en un proceso de múltiples pasos hasta obtener un mapa conceptual que represente fielmente su significado. El objeto principal de este modulo es `CMapCreationWorkflow` el cual coordina la lógica de creación. Por suerte para nosotros, la librería disponde de una manera sencilla de crear objetos de este tipo mediante la clase `WorkflowFactory`. En el siguiente trozo de código crearemos el objeto de flujo de trabajo y escribiremos el texto que queremos interpretar. 

In [44]:
from llms_kgs.logic.cmapper import WorkflowFactory
from llms_kgs.llms import Gemini25Flash
from llms_kgs.domain import Chunk

# Define the knowledge text: 
text = """Leveraging generative Artificial Intelligence (AI), we have transformed \
a dataset comprising 1000 scientific papers focused on biological materials into a \
comprehensive ontological knowledge graph."""

# Create the chunk:
chunk = Chunk(text=text)

# Create the LLM instance: 
llm = Gemini25Flash()

# Create the cmap construction workflow object:
workflow = WorkflowFactory.create_default(llm)

Ahora ejecutaremos el flujo de trabajo para crear el mapa conceptual. El número de mejoras que podemos realizarle a un mapa puede configurarse a gusto. El objeto devuelto es del tipo `CMapCreationState` y contiene los resultados intermedios de cada etapa del proceso de interpretación de texto. 

In [14]:
# Construct the concept map with two improvements: 
cmap_evolution = workflow.create_cmap(chunk, 2)

Imprimimos cada una de las respuestas que brindó el LLM para crear el mapa conceptual:

In [15]:
for data in cmap_evolution.llm_invocations:
    print(data.raw_answer)
    print("="*80)

How was a dataset of scientific papers transformed into an ontological knowledge graph using generative AI?
Generative Artificial Intelligence (AI)
Dataset of Scientific Papers
Biological Materials
Ontological Knowledge Graph
Transformation
Leveraging
transformed into
comprising
focused on
@! Dataset of Scientific Papers @ transformed into @ Ontological Knowledge Graph !@
@! Transformation @ Leveraging @ Generative Artificial Intelligence (AI) !@
@! Dataset of Scientific Papers @ focused on @ Biological Materials !@
@! Dataset of Scientific Papers @ is transformed into @ Ontological Knowledge Graph !@
@! Dataset transformation @ leverages @ Generative Artificial Intelligence (AI) !@
@! Dataset of Scientific Papers @ focuses on @ Biological Materials !@
@! Dataset of Scientific Papers @ comprises @ 1000 scientific papers !@
@! Ontological Knowledge Graph @ is characterized as @ comprehensive !@
@! Dataset of Scientific Papers @ is transformed into @ Ontological Knowledge Graph !@
@! Dat

Tambien podemos leer cada uno de los prompts utilizados para invocar al modelo, tanto de sistema como de usuario. A continuación mostramos los prompts correspondientes a la extracción de ternas de conocimiento.

In [16]:
print(cmap_evolution.llm_invocations[3].system_prompt)
print('='*80)
print(cmap_evolution.llm_invocations[3].user_prompt)

You are a concept map creator who analyzes scientific texts and extracts from them the most important concepts and relationships. A concept is a pattern or regularity in objects, events, or records of objects and events, designated with a label. Concepts are related to each other through meaningful phrases forming propositions. You receive a knowledge text delimited by @ with the following structure:

 1. Focus Question: A question you must answer using Knowledge Triples.
 2. Concept List: A list of relevant concept labels for the Focus Question, which you must use to build the Knowledge Triples.
 3. Semantic Relation List: A list of relevant semantic relationships that you should use to connect concepts and build Knowledge Triples.
 4. Knowledge: A text that answers the Focus Question using the concepts from the Concept List. The Knowledge Triples must be explicitly derived from this text.

Your task is to return a list of Knowledge Triples that answer the Focus Question. A Knowledge 

También podemos graficar los diferentes mapas conceptuales obtenidos. A continuación mostramos el último mapa producido: 

In [45]:
network = CMapDrawer.draw([cmap_evolution.to_cmap(2)])

# Set the height and width of the network:
network.height = "500px"
network.width = "100%"

# Prepare the notebook resources to initialize the template engine
network.prep_notebook() 

# Show the network:
network.show('network.html', notebook=True)

network.html


## Conclusiones

Aprendimos a usar algunas de las herramientas que provee la librería para crear y manipular mapas conceptuales. Estas son las componentes básicas con las cuales crear aplicaciones más avanzadas como sistemas de recuperación de información o de organización de conocimiento. Aunque no lo hemos demostrado aquí, podemos realizar busquedas semánticas sobre el espacio de mapas conceptuales y de ternas de conocimiento, lo que nos permitiría crear sistemas expertos poderosos. También podemos combinar los mapas conceptuales con nuestra arquitectura de chunks para organizar el conocimiento de forma híbrida, aprovechando tanto el texto libre como representaciones estructuradas. 

## Referencias

1. *SQuAD: 100,000+ Questions for Machine Comprehension of Text*. Pranav Rajpurkar, Jian Zhang, Konstantin Lopyrev, Percy Liang.
2. *PubMedQA: A Dataset for Biomedical Research Question Answering*. Qiao Jin, Bhuwan Dhingra, Zhengping Liu, William W. Cohen, Xinghua Lu.
3. *Accelerating Scientific Discovery with Generative Knowledge Extraction, Graph-Based Representation, and Multimodal Intelligent Graph Reasoning*. Markus J. Buhler.
4. *Learning, Creating, and Using Knowledge: Concept Maps as Facilitative Tools in Schools and Corporations*. Joseph D. Novak. 

In [ ]:
from llms_kgs.logic.cmapper import WorkflowFactory
from llms_kgs.llms import Gemini25Flash
from llms_kgs.domain import Chunk

# Define the knowledge text: 
text = """Leveraging generative Artificial Intelligence (AI), we have transformed \
a dataset comprising 1000 scientific papers focused on biological materials into a \
comprehensive ontological knowledge graph."""

# Create the chunk:
chunk = Chunk(text=text)

# Create the LLM instance: 
llm = Gemini25Flash()

# Create the cmap construction workflow object:
workflow = WorkflowFactory.create_default(llm)